In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import math

C:\Users\apple\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [29]:
df = pd.read_csv("final_dataset_hourly.csv", index_col=0, parse_dates=True)

# 填充 UVI > 20 为 NaN
df.loc[df["UVI"] > 20, "UVI"] = np.nan
# 线性插值补全
df = df.interpolate(method="linear")
# 夜间缺失的可填 0
df["UVI"].fillna(0, inplace=True)

# 周期特征编码
df['hour_sin'] = np.sin(2*np.pi*df.index.hour/24)
df['hour_cos'] = np.cos(2*np.pi*df.index.hour/24)
df['day_year_sin'] = np.sin(2*np.pi*df.index.dayofyear/365)
df['day_year_cos'] = np.cos(2*np.pi*df.index.dayofyear/365)

feature_cols = ['temp','rainfall','windspd','windspd_max','wind_d',
                'GHI','DNI','DHI','UVA','ClearSkyGHI','CSI','UVI',
                'hour_sin','hour_cos','day_year_sin','day_year_cos']
target_col = 'UVI'

# 数据集切分
train_df = df.loc['2020-01-01':'2021-06-30']
val_df   = df.loc['2021-07-01':'2021-09-30']
test_df  = df.loc['2021-10-01':'2021-12-29']

C:\Users\apple\AppData\Local\Temp\ipykernel_12888\2944346577.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["UVI"].fillna(0, inplace=True)


In [30]:
x_scaler = StandardScaler()
y_scaler = StandardScaler()

train_x = x_scaler.fit_transform(train_df[feature_cols])
val_x   = x_scaler.transform(val_df[feature_cols])
test_x  = x_scaler.transform(test_df[feature_cols])

train_y = y_scaler.fit_transform(train_df[[target_col]])
val_y   = y_scaler.transform(val_df[[target_col]])
test_y  = y_scaler.transform(test_df[[target_col]])

In [31]:
train_dataset = WeatherTimeSeriesDataset(
    train_x,
    train_y,
    seq_len=96,
    pred_len=24
)

val_dataset = WeatherTimeSeriesDataset(
    val_x,
    val_y,
    seq_len=96,
    pred_len=24
)

test_dataset = WeatherTimeSeriesDataset(
    test_x,
    test_y,
    seq_len=96,
    pred_len=24
)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

13004
2089
2022


In [32]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print(len(train_loader))
print(len(val_loader))
print(len(test_loader))

407
66
64


In [33]:
import torch
from torch.utils.data import Dataset

class WeatherTimeSeriesDataset(Dataset):
    def __init__(self, data_matrix, target_matrix, seq_len=96, pred_len=24):
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.valid_length = len(data_matrix) - seq_len - pred_len + 1
        assert self.valid_length > 0, "数据长度不足"

        self.x = torch.FloatTensor(data_matrix)
        self.y = torch.FloatTensor(target_matrix)

    def __len__(self):
        return self.valid_length

    def __getitem__(self, idx):
        seq_x = self.x[idx : idx + self.seq_len]
        seq_y = self.y[idx + self.seq_len : idx + self.seq_len + self.pred_len, 0]
        return seq_x, seq_y

In [38]:
UVI_idx = feature_cols.index("UVI")

In [39]:
import torch.nn as nn
import torch
import torch.nn.functional as F

class TSMixer(nn.Module):

    def __init__(
        self,
        seq_len=96,
        pred_len=24,
        n_features=15,
        ff_dim=64,
        n_blocks=8,
        dropout=0.1,
        target_idx=UVI_idx
    ):
        super().__init__()

        self.target_idx = target_idx

        self.blocks = nn.ModuleList([
            MixerBlock(
                seq_len,
                n_features,
                ff_dim,
                dropout
            )
            for _ in range(n_blocks)
        ])

        self.head = nn.Linear(
            seq_len,
            pred_len
        )

    def forward(self,x):

        for block in self.blocks:
            x = block(x)

        # 取UVI通道
        x = x[:,:,self.target_idx]

        # 96 -> 24
        x = self.head(x)

        return x

In [40]:
def train_tsmixer(model, train_loader, val_loader, epochs=50, device='cpu', patience=7, lr=1e-3):
    model.to(device)
    criterion = nn.SmoothL1Loss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    best_val_loss = float('inf')
    counter = 0
    best_model = None

    for epoch in range(epochs):
        # --- 训练 ---
        model.train()
        train_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()
            train_loss += loss.item() * batch_x.size(0)
        train_loss /= len(train_loader.dataset)

        # --- 验证 ---
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                pred = model(batch_x)
                loss = criterion(pred, batch_y)
                val_loss += loss.item() * batch_x.size(0)
        val_loss /= len(val_loader.dataset)

        print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model.state_dict()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping (best val={best_val_loss:.4f})")
                break

    model.load_state_dict(best_model)
    return model

In [41]:
def evaluate_tsmixer(model, test_loader, device='cpu'):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            pred = model(batch_x)
            preds.append(pred.cpu())
            trues.append(batch_y.cpu())
    preds = torch.cat(preds, dim=0).numpy()
    trues = torch.cat(trues, dim=0).numpy()
    return trues, preds

In [42]:
device = "cpu"

model = TSMixer(
    seq_len=96,
    pred_len=24,
    n_features=len(feature_cols),  # 15
    ff_dim=64,
    n_blocks=8,
    dropout=0.1
)

model = train_tsmixer(
    model,
    train_loader,
    val_loader,
    epochs=50,
    device=device,
    patience=8,
    lr=1e-3
)

Epoch 01 | Train Loss: 0.1174 | Val Loss: 0.1836
Epoch 02 | Train Loss: 0.0918 | Val Loss: 0.1596
Epoch 03 | Train Loss: 0.0831 | Val Loss: 0.1547
Epoch 04 | Train Loss: 0.0795 | Val Loss: 0.1541
Epoch 05 | Train Loss: 0.0776 | Val Loss: 0.1537
Epoch 06 | Train Loss: 0.0765 | Val Loss: 0.1542
Epoch 07 | Train Loss: 0.0755 | Val Loss: 0.1551
Epoch 08 | Train Loss: 0.0745 | Val Loss: 0.1575
Epoch 09 | Train Loss: 0.0737 | Val Loss: 0.1581
Epoch 10 | Train Loss: 0.0732 | Val Loss: 0.1589
Epoch 11 | Train Loss: 0.0723 | Val Loss: 0.1597
Epoch 12 | Train Loss: 0.0717 | Val Loss: 0.1600
Epoch 13 | Train Loss: 0.0711 | Val Loss: 0.1608
Early stopping (best val=0.1537)


In [43]:
y_true, y_pred = evaluate_tsmixer(model, test_loader, device=device)

# 反归一化
y_true_inv = y_scaler.inverse_transform(y_true.reshape(-1,1)).reshape(y_true.shape)
y_pred_inv = y_scaler.inverse_transform(y_pred.reshape(-1,1)).reshape(y_pred.shape)

In [44]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_true_inv.flatten(), y_pred_inv.flatten())
rmse = np.sqrt(mean_squared_error(y_true_inv.flatten(), y_pred_inv.flatten()))
r2 = r2_score(y_true_inv.flatten(), y_pred_inv.flatten())

print("TSMixer Test (Original Scale)")
print(f"MAE  = {mae:.4f}")
print(f"RMSE = {rmse:.4f}")
print(f"R²   = {r2:.4f}")

TSMixer Test (Original Scale)
MAE  = 0.5888
RMSE = 0.8055
R²   = 0.7878


In [21]:
uvi_df = df[['time', 'UVI']]

uvi_df.to_csv(
    'test_uvi_only.csv',
    index=False
)

['temp', 'rainfall', 'windspd', 'windspd_max', 'wind_d', 'GHI', 'DNI', 'DHI', 'UVA', 'ClearSkyGHI', 'CSI', 'hour_sin', 'hour_cos', 'day_year_sin', 'day_year_cos']
False


In [47]:
time_index = []

for i in range(len(y_pred_inv)):
    start = test_df.index[i + 96]
    rng = pd.date_range(
        start=start,
        periods=24,
        freq='H'
    )
    time_index.extend(rng)

len(time_index)

C:\Users\apple\AppData\Local\Temp\ipykernel_12888\1497663865.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  rng = pd.date_range(


48528

In [49]:
export_df = pd.DataFrame({
    "time": time_index,
    "True_UVI": y_true_inv.flatten(),
    "SunCastNet_Pred": y_pred_inv.flatten()
})

export_df.to_csv(
    "TSMixer_with_time.csv",
    index=False
)